# Chatterbox TTS — Dharmendra voice clone (EN + Hindi) on Colab

Runs the CPU-only chatterbox-tts server with the Dharmendra voice clones from
[github.com/Dkpiec/chatterbox-tts](https://github.com/Dkpiec/chatterbox-tts), then exposes it
through a public HTTPS tunnel.

**Setup:**
1. Runtime → Change runtime type → **CPU** (the server is CPU-only; GPU wastes quota)
2. Runtime → **Run all**

**Timings:** ~20 min first run (install + first voiceover downloads ~8 GB of models).
Later runs with the Drive cache enabled: ~5 min.

**Limits:** free Colab sessions die after ~90 min idle / 12 h max, and the tunnel URL changes
every run. For an always-on endpoint use your own server instead.

In [ ]:
# 1) Config + optional Google Drive model cache
USE_DRIVE_CACHE = True
DRIVE_CACHE_DIR = "/content/drive/MyDrive/chatterbox_tts_cache"
PORT = 8001

import os
os.makedirs("/content/hf_cache", exist_ok=True)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(f"{DRIVE_CACHE_DIR}/hf_cache"):
        print("Restoring HF cache from Drive...")
        !cp -rn "{DRIVE_CACHE_DIR}/hf_cache/." /content/hf_cache/ 2>/dev/null
        n = sum(len(fs) for _, _, fs in os.walk("/content/hf_cache"))
        print(f"restored ({n} files)")
    else:
        print("No Drive cache yet — models will download on first voiceover, then cell 7 saves them.")

In [ ]:
# 2) Install chatterbox-tts — pinned torch trio (fixes the HTTP 500)
# chatterbox-tts pins torch 2.6, but Colab ships torchvision 0.26 built for torch 2.11.
# Mixing them crashes with "operator torchvision::nms does not exist" -> HTTP 500 on /tts.
# Fix: install the exact matching torch/torchvision/torchaudio CPU trio first.
!apt-get -qq install -y libgomp1 > /dev/null 2>&1

import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd); print(r.stdout[-800:]); print(r.stderr[-800:])
    return r.returncode

sh("pip install -q torch==2.6.0+cpu torchvision==0.21.0+cpu torchaudio==2.6.0+cpu --index-url https://download.pytorch.org/whl/cpu")
sh("pip install -q chatterbox-tts==0.1.7")

# sanity check — these imports are exactly what crashed before
import torch
print("torch:", torch.__version__)
import torchaudio
print("torchaudio:", torchaudio.__version__)
try:
    import torchvision
    print("torchvision:", torchvision.__version__)
except Exception as e:
    print("torchvision unusable:", repr(e), "-> uninstalling (chatterbox does not need vision)")
    sh("pip uninstall -y -q torchvision")
print("installed")

In [ ]:
# 3) Get the server + voice refs, wire the Dharmendra clones, start the server
import os, json, subprocess, time, urllib.request

os.chdir("/content")
if not os.path.exists("/content/chatterbox-tts/server.py"):
    !git clone -q https://github.com/Dkpiec/chatterbox-tts.git

with open("/content/chatterbox-tts/current_clone.json", "w") as f:
    json.dump({
        "en":  {"audio_wav": "/content/chatterbox-tts/voice_ref.wav",    "exaggeration": 0.4},
        "mtl": {"audio_wav": "/content/chatterbox-tts/voice_ref_hi.wav", "exaggeration": 0.4},
    }, f, indent=1)

# pre-flight: verify chatterbox imports cleanly (fails fast here instead of HTTP 500 later)
try:
    from chatterbox.tts import ChatterboxTTS  # noqa: F401
    print("chatterbox import OK")
except Exception as e:
    raise RuntimeError(f"PREFLIGHT FAILED — chatterbox cannot import: {e!r}") from e

env = dict(os.environ, HF_HOME="/content/hf_cache", PORT=str(PORT), IDLE_UNLOAD_S="3600")
log = open("/content/chatterbox_server.log", "w")
proc = subprocess.Popen(["python", "/content/chatterbox-tts/server.py"],
                        env=env, stdout=log, stderr=subprocess.STDOUT)
print("server pid:", proc.pid)

for _ in range(30):
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print("health:", r.read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print("server did not respond — check /content/chatterbox_server.log")
    !tail -20 /content/chatterbox_server.log

In [ ]:
# 4) Public HTTPS tunnel (URL changes every run)
import subprocess, time, re
!curl -sLo /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tun_log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                  "--no-autoupdate"], stdout=tun_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"(https://[a-z0-9\-]+\.trycloudflare\.com)", open("/content/tunnel.log").read())
    if m:
        PUBLIC_URL = m.group(1)
        break
print("PUBLIC URL:", PUBLIC_URL)
if PUBLIC_URL:
    print("Health check:", PUBLIC_URL + "/health")

In [ ]:
# 5) First voiceover test — English.
# FIRST CALL ONLY: downloads the English model (~2-5 min), then loads it (~2 min). Be patient.
# If it fails, the server log is printed below for diagnosis.
import json, urllib.request, traceback
from IPython.display import Audio, display

body = json.dumps({"text": "Hello, this is a test of the cloned voice. It should sound like Dharmendra.",
                   "lang": "en", "exaggeration": 0.4}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/tts", data=body,
                             headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=1800) as r:
        wav = r.read()
    open("/content/test_en.wav", "wb").write(wav)
    print(f"{len(wav)} bytes")
    display(Audio("/content/test_en.wav"))
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    print("\n=== SERVER LOG (last 40 lines) ===")
    !tail -40 /content/chatterbox_server.log
    print("\n=== DIAGNOSIS ===")
    print("Common causes: model download failed (network/HF rate limit), OOM, or missing voice ref.")
    print("Try: re-run this cell. If it persists, copy the log above and share it.")
except Exception as ex:
    print(f"Error: {ex}")
    traceback.print_exc()
    print("\n=== SERVER LOG (last 40 lines) ===")
    !tail -40 /content/chatterbox_server.log

In [ ]:
# 6) Generate your own text and download the wav
# First Hindi call downloads the multilingual model (~5GB, a few minutes) and switches models.
# Switching en<->hi later takes ~2 min per switch.
from google.colab import files

TEXT = "नमस्ते, यह धर्मेंद्र की आवाज़ का टेस्ट है।"
LANG = "hi"   # 'en' = English, 'hi' = Hindi (22 more languages supported)

body = json.dumps({"text": TEXT, "lang": LANG, "exaggeration": 0.4}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/tts", data=body,
                             headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=3600) as r:
        wav = r.read()
    out = f"/content/voiceover_{LANG}.wav"
    open(out, "wb").write(wav)
    display(Audio(out))
    files.download(out)
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    print("\n=== SERVER LOG (last 40 lines) ===")
    !tail -40 /content/chatterbox_server.log
except Exception as ex:
    print(f"Error: {ex}")
    !tail -40 /content/chatterbox_server.log

In [ ]:
# 7) When done: save model cache to Drive, then stop everything.
if USE_DRIVE_CACHE:
    os.makedirs(f"{DRIVE_CACHE_DIR}/hf_cache", exist_ok=True)
    print("Saving HF cache to Drive — first save of ~8GB takes several minutes...")
    !cp -rn /content/hf_cache/. "{DRIVE_CACHE_DIR}/hf_cache/"
    print("saved")
!pkill -f "chatterbox-tts/server.py" || true
!pkill -f cloudflared || true
print("stopped")